In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import  PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import  InMemoryVectorStore
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field

In [ ]:
## Document load
loader=PyPDFLoader("../data/2026.pdf")
docs=loader.load()

## split -chunks
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs=splitter.split_documents(docs)

## embedding and vector db
embeddings=OpenAIEmbeddings(model="text-embedding-3-large")
vector_store=InMemoryVectorStore.from_documents(
    documents=docs,
    embedding=embeddings)

In [ ]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
 


class RagState(BaseModel):
    question:str = Field(description="user questions")
    documents: list= []
    context: str = Field(description="context data for user question",default="")
    answer: str = Field(description=" Final answer...",default="")

In [ ]:
#question -> retrieve ->context->generate->end

def retrieve(state:RagState):
    docs=vector_store.similarity_search(query=state.question)
    state.documents=docs
    return state

def create_context(state:RagState) -> RagState:
    context=""
    for doc in state.documents:
        context+=doc.page_content + "\n"
    state.context=context
    return state
    
def generate_node(state:RagState)->RagState:
    prompt=f"""you are a assistant and provide the answer for user question based on the provided context.
    if you dont find the releveangt answer, then just say 'i dont know'
    cotext is:{state.context}
    questions is:{state.question}"""
    res=llm.invoke(prompt)
    state.answer=res.content
    return state

In [ ]:
graph=StateGraph(RagState)
graph.add_node("retrieve",retrieve)
graph.add_node("create_context",create_context)
graph.add_node("generate",generate_node)
graph.add_edge(START,'retrieve_node')
graph.add_edge('retrieve_node','create_context_node')
graph.add_edge('create_context_node','generte_node')

graph.add_edge('generate_node',END)

graph=graph.compile()



In [ ]:
from Ipython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
res=graph.invoke({"question":"what is the name of singer"})

In [ ]:
print(res["answer"])